In [94]:
import pandas as pd
import numpy as np

# -------------------------------
# 1. LOAD CSV FILES
# -------------------------------
df = pd.read_csv("final_resilience_data.csv")
em = pd.read_csv("emat_disasters.csv")

In [96]:
df

,Country Name,Country Code,year,gdp_current_usd,gdp_per_capita_usd,population,gdp_growth_pct,iso3,region,country_disaster,...,total_damage_usd,country_hdi,hdi,life_expectancy,fatalities_per_1m,affected_pct,damage_pct_gdp,DII,CRI,RRS_Proxy
0,Aruba,ABW,2000,1.873453e+09,20681.023027,90588.0,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN
1,Aruba,ABW,2001,1.896457e+09,20740.132583,91439.0,1.227918,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN
2,Aruba,ABW,2002,1.961844e+09,21307.248251,92074.0,3.447829,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN
3,Aruba,ABW,2003,2.044112e+09,21949.485996,93128.0,4.193411,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN
4,Aruba,ABW,2004,2.254831e+09,23700.631990,95138.0,10.308585,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6620,Zimbabwe,ZWE,2020,2.686856e+10,1730.453910,15526888.0,4.483288,NaN,NaN,NaN,...,0.0,Zimbabwe,0.598,62.8,0.000000,0.000000,0.000000,0.000000,5980.000000,5.231644
6621,Zimbabwe,ZWE,2021,2.724051e+10,1724.387271,15797210.0,1.384308,ZWE,Africa,Zimbabwe,...,0.0,Zimbabwe,0.598,62.8,0.189907,1.075506,0.000000,0.000734,299.000000,3.682154
6622,Zimbabwe,ZWE,2022,3.278966e+10,2040.546587,16069056.0,20.370947,ZWE,Africa,Zimbabwe,...,0.0,Zimbabwe,0.598,62.8,46.673557,0.059437,0.000000,0.022902,199.333333,13.175474
6623,Zimbabwe,ZWE,2023,3.523137e+10,2156.034093,16340822.0,7.446592,ZWE,Africa,Zimbabwe,...,0.0,Zimbabwe,0.598,62.8,44.122627,0.025807,0.000000,0.020477,299.000000,6.713296


In [97]:
df.isna().sum()

Country Name             0
Country Code             0
year                     0
gdp_current_usd        212
gdp_per_capita_usd     212
population               0
gdp_growth_pct         478
iso3                  3754
region                3754
country_disaster      3754
events_count             0
total_deaths             0
total_affected           0
total_damage_usd         0
country_hdi           2425
hdi                   2425
life_expectancy       2425
fatalities_per_1m        0
affected_pct             0
damage_pct_gdp         212
DII                    212
CRI                   2471
RRS_Proxy             2639
dtype: int64

In [98]:
em = em.rename(columns={
    'ISO': 'iso3',
    'Country': 'Country Name',
    'Start Year': 'year',
    'Total Deaths': 'total_deaths',
    'Total Affected': 'total_affected',
    "Total Damage ('000 US$)": 'total_damage_usd',
    'Disaster Type': 'disaster_type',
    'Disaster Subtype': 'disaster_subtype',
    'Disaster Group': 'disaster_group',
    'Disaster Subgroup': 'disaster_subgroup'
})

In [99]:
# Aggregate numeric columns
em_agg = em.groupby(['Country Name', 'year']).agg({
    'total_deaths': 'sum',
    'total_affected': 'sum',
    'total_damage_usd': 'sum',
    'disaster_type': lambda x: x.mode()[0] if not x.mode().empty else None,
    'disaster_subtype': lambda x: x.mode()[0] if not x.mode().empty else None,
    'disaster_group': lambda x: x.mode()[0] if not x.mode().empty else None,
    'disaster_subgroup': lambda x: x.mode()[0] if not x.mode().empty else None
}).reset_index()


In [100]:
df = df.merge(em_agg, how='left', on=['Country Name', 'year'])


In [118]:
df.columns

Index(['Country Name', 'Country Code', 'year', 'gdp_current_usd',
       'gdp_per_capita_usd', 'population', 'gdp_growth_pct', 'iso3', 'region',
       'country_disaster', 'events_count', 'country_hdi', 'hdi',
       'life_expectancy', 'fatalities_per_1m', 'affected_pct',
       'damage_pct_gdp', 'DII', 'CRI', 'RRS_Proxy', 'disaster_type',
       'disaster_subtype', 'disaster_group', 'disaster_subgroup',
       'total_deaths', 'total_affected', 'total_damage_usd', 'severity'],
      dtype='object')

In [103]:
df['total_deaths'] = df['total_deaths_x'].fillna(df['total_deaths_y'])
df['total_affected'] = df['total_affected_x'].fillna(df['total_affected_y'])
df['total_damage_usd'] = df['total_damage_usd_x'].fillna(df['total_damage_usd_y'])

# Drop the temporary columns
df = df.drop(columns=['total_deaths_x', 'total_deaths_y',
                      'total_affected_x', 'total_affected_y',
                      'total_damage_usd_x', 'total_damage_usd_y'])

df = df.merge(em_agg, how='left', on=['Country Name','year'], 
              suffixes=('', '_em'))

df['total_deaths'] = df['total_deaths'].fillna(df['total_deaths_em'])
df['total_affected'] = df['total_affected'].fillna(df['total_affected_em'])
df['total_damage_usd'] = df['total_damage_usd'].fillna(df['total_damage_usd_em'])

df = df.drop(columns=[c for c in df.columns if c.endswith('_em')])


In [116]:
df.isna().sum()

Country Name             0
Country Code             0
year                     0
gdp_current_usd          0
gdp_per_capita_usd       0
population               0
gdp_growth_pct         478
iso3                  3754
region                3754
country_disaster      3754
events_count             0
country_hdi           6625
hdi                      0
life_expectancy          0
fatalities_per_1m        0
affected_pct             0
damage_pct_gdp           0
DII                      0
CRI                   2471
RRS_Proxy             2639
disaster_type            0
disaster_subtype         0
disaster_group           0
disaster_subgroup        0
total_deaths             0
total_affected           0
total_damage_usd         0
severity                 0
dtype: int64

In [106]:
disaster_cols = ['disaster_type', 'disaster_subtype', 'disaster_group', 'disaster_subgroup']
for col in disaster_cols:
    df[col] = df[col].fillna('No Disaster')


In [109]:
# gdp_current_usd
df['gdp_current_usd'] = df.groupby('iso3')['gdp_current_usd'].transform(
    lambda x: x.interpolate(limit_direction='both')
)
df['gdp_current_usd'] = df['gdp_current_usd'].fillna(df.groupby('region')['gdp_current_usd'].transform('median'))
df['gdp_current_usd'] = df['gdp_current_usd'].fillna(df['gdp_current_usd'].median())

# gdp_per_capita_usd
df['gdp_per_capita_usd'] = df['gdp_per_capita_usd'].fillna(
    df['gdp_current_usd'] / df['population']
)


In [111]:
# Ensure numeric columns are numeric
for col in ['hdi', 'country_hdi', 'life_expectancy']:
    if col in df.columns:
        # Convert to numeric, invalid parsing becomes NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Fill missing values by country
        df[col] = df.groupby('iso3')[col].transform(lambda x: x.ffill().bfill())
        # Fill remaining missing with global median
        df[col] = df[col].fillna(df[col].median())


In [115]:
df['damage_pct_gdp'] = df['damage_pct_gdp'].fillna(
    df['total_damage_usd'] / df['gdp_current_usd'] * 100
)

severity_map = {
    'Earthquake': 2.0,
    'Flood': 1.0,
    'Drought': 1.2,
    'Storm': 1.5,
    'Wildfire': 1.1,
    'No Disaster': 1.0
}

df['severity'] = df['disaster_type'].map(severity_map).fillna(1.0)

df['DII'] = df['DII'].fillna(
    (df['fatalities_per_1m'] + df['affected_pct']) / df['gdp_per_capita_usd'] * df['severity']
)


In [122]:
df['Exposure'] = df.groupby('iso3')['events_count'].transform(lambda x: x / x.max())
df['Vulnerability'] = df['fatalities_per_1m'] + df['damage_pct_gdp']
df['AdaptiveCapacity'] = df['hdi'].fillna(df['hdi'].median())


df['RRS_Proxy'] = df['RRS_Proxy'].fillna(df['gdp_growth_pct'] / (1 + df['fatalities_per_1m']))


In [ ]:
df['CRI'] = df['CRI'].fillna(df['AdaptiveCapacity'] / (df['Exposure'] * df['Vulnerability']))
df['CRI'] = 100 * (df['CRI'] - df['CRI'].min()) / (df['CRI'].max() - df['CRI'].min())

In [131]:
print("Final missing values per column:")
print(df.isna().sum())


Final missing values per column:
Country Name             0
Country Code             0
year                     0
gdp_current_usd          0
gdp_per_capita_usd       0
population               0
gdp_growth_pct           0
iso3                  3754
region                3754
country_disaster      3754
events_count             0
country_hdi              0
hdi                      0
life_expectancy          0
fatalities_per_1m        0
affected_pct             0
damage_pct_gdp           0
DII                      0
CRI                   2471
RRS_Proxy                0
disaster_type            0
disaster_subtype         0
disaster_group           0
disaster_subgroup        0
total_deaths             0
total_affected           0
total_damage_usd         0
severity                 0
Exposure                 0
Vulnerability            0
AdaptiveCapacity         0
dtype: int64


In [124]:
df['gdp_growth_pct'] = df['gdp_growth_pct'].fillna(0)
# OR (safer)
df['gdp_growth_pct'] = df['gdp_growth_pct'].fillna(df.groupby('region')['gdp_growth_pct'].transform('median'))


In [87]:
df

,Country Name,Country Code,year,gdp_current_usd,gdp_per_capita_usd,population,gdp_growth_pct,iso3,region,country_disaster,...,disaster_subtype,disaster_group,disaster_subgroup,total_deaths,total_affected,total_damage_usd,severity,Exposure,Vulnerability,AdaptiveCapacity
0,Aruba,ABW,2000,4.686947e+10,20681.023027,90588.0,0.000000,NaN,NaN,NaN,...,No Disaster,No Disaster,No Disaster,0.0,0.0,0.0,1.0,NaN,0.000000,0.756
1,Aruba,ABW,2001,4.686947e+10,20740.132583,91439.0,1.227918,NaN,NaN,NaN,...,No Disaster,No Disaster,No Disaster,0.0,0.0,0.0,1.0,NaN,0.000000,0.756
2,Aruba,ABW,2002,4.686947e+10,21307.248251,92074.0,3.447829,NaN,NaN,NaN,...,No Disaster,No Disaster,No Disaster,0.0,0.0,0.0,1.0,NaN,0.000000,0.756
3,Aruba,ABW,2003,4.686947e+10,21949.485996,93128.0,4.193411,NaN,NaN,NaN,...,No Disaster,No Disaster,No Disaster,0.0,0.0,0.0,1.0,NaN,0.000000,0.756
4,Aruba,ABW,2004,4.686947e+10,23700.631990,95138.0,10.308585,NaN,NaN,NaN,...,No Disaster,No Disaster,No Disaster,0.0,0.0,0.0,1.0,NaN,0.000000,0.756
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6620,Zimbabwe,ZWE,2020,4.686947e+10,1730.453910,15526888.0,4.483288,NaN,NaN,NaN,...,No Disaster,No Disaster,No Disaster,0.0,0.0,0.0,1.0,NaN,0.000000,0.756
6621,Zimbabwe,ZWE,2021,2.724051e+10,1724.387271,15797210.0,1.384308,ZWE,Africa,Zimbabwe,...,Drought,Natural,Climatological,3.0,169900.0,0.0,1.2,0.4,0.189907,0.598
6622,Zimbabwe,ZWE,2022,3.278966e+10,2040.546587,16069056.0,20.370947,ZWE,Africa,Zimbabwe,...,Flash flood,Natural,Biological,750.0,9551.0,0.0,1.0,0.6,46.673557,0.598
6623,Zimbabwe,ZWE,2023,3.523137e+10,2156.034093,16340822.0,7.446592,ZWE,Africa,Zimbabwe,...,Bacterial disease,Natural,Biological,721.0,4217.0,0.0,1.0,0.4,44.122627,0.598


In [126]:
df['country_hdi'] = df['country_hdi'].fillna(df['hdi'])
df['country_hdi'] = df['country_hdi'].fillna(df['country_hdi'].median())


In [128]:
df['Exposure'] = df.groupby('iso3')['events_count'].transform(lambda x: x / x.max())
df['Exposure'] = df['Exposure'].fillna(0)  # no events => Exposure = 0


In [130]:
df['RRS_Proxy'] = df['RRS_Proxy'].fillna(df['gdp_growth_pct'] / (1 + df['fatalities_per_1m']))


In [135]:
df_clean['CRI']

0        38.387327
1       155.059066
2        31.000000
3        55.111111
4       165.333333
           ...    
2415    299.000000
2416    299.000000
2417    199.333333
2418    299.000000
2419      0.264242
Name: CRI, Length: 2420, dtype: float64

In [134]:
# Drop rows where key columns are missing
key_columns = ['iso3', 'region', 'CRI']  # You can add others if needed
df_clean = df.dropna(subset=key_columns).reset_index(drop=True)

# Confirm no missing values in key columns
print("Missing values after dropping rows:")
print(df_clean[key_columns].isna().sum())

# Optional: check total rows dropped
rows_dropped = len(df) - len(df_clean)
print(f"Rows dropped: {rows_dropped}")


Missing values after dropping rows:
iso3      0
region    0
CRI       0
dtype: int64
Rows dropped: 4205


In [136]:
df_clean.isna().sum()
df_clean

,Country Name,Country Code,year,gdp_current_usd,gdp_per_capita_usd,population,gdp_growth_pct,iso3,region,country_disaster,...,disaster_subtype,disaster_group,disaster_subgroup,total_deaths,total_affected,total_damage_usd,severity,Exposure,Vulnerability,AdaptiveCapacity
0,Afghanistan,AFG,2000,3.521418e+09,174.930991,20130327.0,0.000000,AFG,Asia,Afghanistan,...,Viral disease,Natural,Biological,594.0,2582228.0,91000.0,1.0,0.3125,29.510302,0.496
1,Afghanistan,AFG,2001,2.813572e+09,138.706822,20284307.0,-20.101172,AFG,Asia,Afghanistan,...,Cold wave,Natural,Biological,485.0,204695.0,18000.0,1.0,0.3125,23.910749,0.496
2,Afghanistan,AFG,2002,3.825701e+09,178.954088,21378117.0,35.973125,AFG,Asia,Afghanistan,...,Infectious disease (General),Natural,Biological,4083.0,313670.0,0.0,1.0,1.0000,190.989693,0.496
3,Afghanistan,AFG,2003,4.520947e+09,198.871116,22733049.0,18.173017,AFG,Asia,Afghanistan,...,Flash flood,Natural,Hydrological,137.0,4754.0,0.0,1.0,0.5625,6.026468,0.496
4,Afghanistan,AFG,2004,5.224897e+09,221.763654,23560654.0,15.570851,AFG,Asia,Afghanistan,...,Riverine flood,Natural,Hydrological,18.0,5540.0,0.0,1.0,0.1875,0.763986,0.496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2415,Zimbabwe,ZWE,2019,2.571566e+10,1683.913136,15271368.0,-24.711284,ZWE,Africa,Zimbabwe,...,Flash flood,Natural,Hydrological,654.0,270486.0,0.0,1.0,0.4000,42.825240,0.598
2416,Zimbabwe,ZWE,2021,2.724051e+10,1724.387271,15797210.0,1.384308,ZWE,Africa,Zimbabwe,...,Drought,Natural,Climatological,3.0,169900.0,0.0,1.2,0.4000,0.189907,0.598
2417,Zimbabwe,ZWE,2022,3.278966e+10,2040.546587,16069056.0,20.370947,ZWE,Africa,Zimbabwe,...,Flash flood,Natural,Biological,750.0,9551.0,0.0,1.0,0.6000,46.673557,0.598
2418,Zimbabwe,ZWE,2023,3.523137e+10,2156.034093,16340822.0,7.446592,ZWE,Africa,Zimbabwe,...,Bacterial disease,Natural,Biological,721.0,4217.0,0.0,1.0,0.4000,44.122627,0.598


In [137]:
# Save final clean dataframe to CSV
df_clean.to_csv("final_clean_disaster_data_2.csv", index=False)

print("Final cleaned CSV saved as 'final_clean_disaster_data.csv'")


Final cleaned CSV saved as 'final_clean_disaster_data.csv'


In [139]:
import numpy as np
import pandas as pd

# use df_clean if that's the dataframe you're using; fallback to df
dfu = df_clean if 'df_clean' in globals() else df

# Ensure numerics
num_cols = ['events_count','total_damage_usd','gdp_current_usd','population','total_deaths','total_affected','gdp_per_capita_usd','gdp_growth_pct','fatalities_per_1m']
for c in num_cols:
    if c in dfu.columns:
        dfu[c] = pd.to_numeric(dfu[c], errors='coerce').fillna(0)

# ------------------------
# R2: Feature engineering
# ------------------------

# 1) Annualized disaster frequency (events_count per year per country)
# Assuming events_count is already a per-country-year count, annualized frequency = events_count
dfu['annual_disaster_freq'] = dfu['events_count']

# 2) Average economic loss per event (normalized by GDP)
# If events_count == 0 -> set average_loss_per_event = 0
dfu['avg_loss_per_event_usd'] = np.where(
    dfu['events_count'] > 0,
    dfu['total_damage_usd'] / dfu['events_count'],
    0.0
)
# normalized by GDP (percentage of GDP per event)
dfu['avg_loss_per_event_pct_gdp'] = np.where(
    dfu['events_count'] > 0,
    (dfu['avg_loss_per_event_usd'] / dfu['gdp_current_usd']) * 100,
    0.0
)

# 3) Recovery rate (based on GDP rebound)
# Requires previous-year GDP to compute rebound; compute simple year-on-year rebound:
dfu = dfu.sort_values(['iso3','year'])
dfu['gdp_prev'] = dfu.groupby('iso3')['gdp_current_usd'].shift(1)
# recovery_ratio = (gdp_current - gdp_prev) / abs(gdp_prev) if gdp_prev < gdp_current (simplified)
dfu['recovery_rate'] = np.nan
mask = dfu['gdp_prev'].notna() & (dfu['gdp_prev'] != 0)
dfu.loc[mask, 'recovery_rate'] = (dfu.loc[mask, 'gdp_current_usd'] - dfu.loc[mask, 'gdp_prev']) / dfu.loc[mask, 'gdp_prev']
# fill remaining with 0
dfu['recovery_rate'] = dfu['recovery_rate'].fillna(0)

# 4) Infrastructure exposure (urbanization × hazard intensity)
# NOTE: requires 'urbanization' and 'hazard_intensity' sources. If you don't have them, create proxies:
# proxy_urbanization = population density approximation if you have area; otherwise NA.
# hazard_intensity proxy: use events_count relative to max for the country or region.
# Here create a simple proxy hazard_intensity = events_count normalized (0-1)
dfu['hazard_intensity_proxy'] = dfu.groupby('iso3')['events_count'].transform(
    lambda x: x / (x.max() if x.max() != 0 else 1)
)

# If you don't have urbanization, you can put a placeholder or get urban% from World Bank.
# We'll create a placeholder 'urban_pct' = NaN (user should replace with real data)
if 'urban_pct' not in dfu.columns:
    dfu['urban_pct'] = np.nan
# infrastructure_exposure proxy
dfu['infrastructure_exposure_proxy'] = dfu['urban_pct'] * dfu['hazard_intensity_proxy']

# 5) Human cost ratio (fatalities per 100k population)
dfu['human_cost_per_100k'] = np.where(
    dfu['population'] > 0,
    dfu['total_deaths'] / dfu['population'] * 100000,
    0.0
)

# ------------------------
# R3: Recompute / explain indices
# ------------------------
# 1) Disaster Impact Index (DII) - example formula
# DII = severity * (fatalities_per_1m + affected_pct) / gdp_per_capita_usd
# severity: map disaster_type to weight
severity_map = {
    'Earthquake': 2.0,
    'Flood': 1.0,
    'Drought': 1.2,
    'Storm': 1.5,
    'Wildfire': 1.1,
    'No Disaster': 1.0,
    'Unknown': 1.0
}
dfu['severity_w'] = dfu['disaster_type'].map(severity_map).fillna(1.0)

# Avoid divide by zero: replace zeros in gdp_per_capita_usd with median
gdp_pc_med = dfu['gdp_per_capita_usd'].replace(0, np.nan).median()
dfu['gdp_per_capita_usd'] = dfu['gdp_per_capita_usd'].replace(0, np.nan).fillna(gdp_pc_med)

dfu['DII_recalc'] = (dfu['fatalities_per_1m'] + dfu['affected_pct']) / dfu['gdp_per_capita_usd'] * dfu['severity_w']

# 2) Community Resilience Index (CRI) - example formula and normalization
# CRI_raw = AdaptiveCapacity / (Exposure * Vulnerability)
# where Exposure in [0,1], Vulnerability = fatalities_per_1m + damage_pct_gdp
dfu['Exposure'] = dfu['Exposure'].fillna(0)
dfu['Vulnerability'] = dfu['fatalities_per_1m'] + dfu['damage_pct_gdp']
# protect division by zero by replacing 0 with small epsilon
eps = 1e-6
dfu['CRI_raw'] = dfu['AdaptiveCapacity'] / (dfu['Exposure'].replace(0, eps) * dfu['Vulnerability'].replace(0, eps))
# Normalize CRI_raw to 0-100 (robust scaling using min/max; consider clipping to remove outliers)
minv, maxv = dfu['CRI_raw'].min(), dfu['CRI_raw'].max()
if pd.isna(minv) or pd.isna(maxv) or minv == maxv:
    dfu['CRI_recalc'] = 50.0
else:
    dfu['CRI_recalc'] = 100 * (dfu['CRI_raw'] - minv) / (maxv - minv)

# 3) Recovery & Resilience Score (RRS_Proxy) - example
# RRS = gdp_growth_pct / (1 + fatalities_per_1m)
dfu['RRS_recalc'] = dfu['gdp_growth_pct'] / (1 + dfu['fatalities_per_1m'])

# ------------------------
# Sanity checks & final cleanup
# ------------------------
# Replace inf/nans produced by div by zero with zeros or medians where appropriate
dfu.replace([np.inf, -np.inf], np.nan, inplace=True)
dfu.fillna({
    'avg_loss_per_event_usd': 0,
    'avg_loss_per_event_pct_gdp': 0,
    'human_cost_per_100k': 0,
    'DII_recalc': 0,
    'CRI_recalc': dfu['CRI_recalc'].median() if 'CRI_recalc' in dfu else 50,
    'RRS_recalc': 0
}, inplace=True)

# drop temp columns if you want
# dfu = dfu.drop(columns=['gdp_prev','severity_w','CRI_raw'])

# Save final CSV (performance-friendly; no index)
out_path = "final_clean_disaster_data_with_features.csv"
dfu.to_csv(out_path, index=False)
print("Saved:", out_path)

# show remaining nulls
print("Remaining null counts (quick):")
print(dfu.isna().sum().sort_values(ascending=False).head(20))


Saved: final_clean_disaster_data_with_features.csv
Remaining null counts (quick):
infrastructure_exposure_proxy    2420
urban_pct                        2420
gdp_prev                          151
avg_loss_per_event_pct_gdp          0
total_affected                      0
total_damage_usd                    0
severity                            0
Exposure                            0
Vulnerability                       0
AdaptiveCapacity                    0
annual_disaster_freq                0
avg_loss_per_event_usd              0
Country Name                        0
disaster_subgroup                   0
recovery_rate                       0
hazard_intensity_proxy              0
human_cost_per_100k                 0
severity_w                          0
DII_recalc                          0
CRI_raw                             0
dtype: int64


In [142]:
dfu.columns

Index(['Country Name', 'Country Code', 'year', 'gdp_current_usd',
       'gdp_per_capita_usd', 'population', 'gdp_growth_pct', 'iso3', 'region',
       'country_disaster', 'events_count', 'country_hdi', 'hdi',
       'life_expectancy', 'fatalities_per_1m', 'affected_pct',
       'damage_pct_gdp', 'DII', 'CRI', 'RRS_Proxy', 'disaster_type',
       'disaster_subtype', 'disaster_group', 'disaster_subgroup',
       'total_deaths', 'total_affected', 'total_damage_usd', 'severity',
       'Exposure', 'Vulnerability', 'AdaptiveCapacity', 'annual_disaster_freq',
       'avg_loss_per_event_usd', 'avg_loss_per_event_pct_gdp', 'gdp_prev',
       'recovery_rate', 'hazard_intensity_proxy', 'urban_pct',
       'infrastructure_exposure_proxy', 'human_cost_per_100k', 'severity_w',
       'DII_recalc', 'CRI_raw', 'CRI_recalc', 'RRS_recalc'],
      dtype='object')